In [3]:
import sys, os
import numpy as np
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
from config import PATH_RAW_DATA, CSV_ENCODING, CSV_SEPARATOR, NA_MARKERS, PATH_GOLD_DATA

pd.set_option('display.max_columns', None)

In [4]:
arquivo = PATH_GOLD_DATA/'madeiras.csv'
df = pd.read_csv(arquivo)

colunas = [
    "densidade_basica",
    "contracao_tangencial",
    "contracao_radial",
    "contracao_volumetrica",
    "relacao_tangencial_radial",
    "flexao_seca_moe",
    "flexao_seca_mor",
    "compressao_paralela_seca",
    "dureza_janka_paralela_seca",
    "dureza_janka_transversal_seca",
    "cisalhamento_seca",
]

# Remove espécies que não possuem todos os valores necessários
base = df.dropna(subset=colunas).copy()

# Normalização z-score:
# transforma todas as colunas para uma escala comparável
media = base[colunas].mean()
desvio = base[colunas].std()

dados_normalizados = (base[colunas] - media) / desvio


def especies_semelhantes(nome_especie, quantidade=5):
    especie = base[base["nome_cientifico"] == nome_especie]

    if especie.empty:
        raise ValueError(f"Espécie não encontrada: {nome_especie}")

    indice_especie = especie.index[0]
    vetor_referencia = dados_normalizados.loc[indice_especie]

    # Distância euclidiana entre a espécie de referência
    # e todas as outras espécies
    diferencas = dados_normalizados - vetor_referencia
    distancias = np.sqrt((diferencas ** 2).sum(axis=1))

    resultado = base[
        [
            "nome_cientifico",
            "nome_popular_1",
            "familia",
            "densidade_basica",
        ]
    ].copy()

    resultado["distancia"] = distancias

    # Remove a própria espécie
    resultado = resultado[resultado["nome_cientifico"] != nome_especie]

    return resultado.sort_values("distancia").head(quantidade)


resultado = especies_semelhantes(
    "Handroanthus serratifolius = Tabebuia serratifolia",
    quantidade=5,
)

print(resultado.to_string(index=False))

                nome_cientifico               nome_popular_1       familia  densidade_basica  distancia
               Bowdichia nitida                     SUCUPIRA      Fabaceae              0.77   1.782615
           Myrocarpus frondosus               CABREÚVA-PARDA      Fabaceae              0.78   1.853681
            Terminalia amazonia                    TANIMBUCA  Combretaceae              0.80   2.202020
              Maclura tinctoria             AMORA / AMOREIRA      Moraceae              0.73   2.457946
Lecythis pisonis subsp. usitata SAPUCAIA / SAPUCAIA-VERMELHA Lecythidaceae              0.84   2.532288


In [5]:
resultado.head(3)

,nome_cientifico,nome_popular_1,familia,densidade_basica,distancia
2,Bowdichia nitida,SUCUPIRA,Fabaceae,0.77,1.782615
19,Myrocarpus frondosus,CABREÚVA-PARDA,Fabaceae,0.78,1.853681
240,Terminalia amazonia,TANIMBUCA,Combretaceae,0.80,2.202020


In [7]:
pesos = {
    "densidade_basica": 1.0,
    "contracao_tangencial": 1.0,
    "contracao_radial": 1.0,
    "contracao_volumetrica": 1.0,
    "relacao_tangencial_radial": 1.0,
    "flexao_seca_moe": 1.5,
    "flexao_seca_mor": 1.5,
    "compressao_paralela_seca": 1.5,
    "dureza_janka_paralela_seca": 2.0,
    "dureza_janka_transversal_seca": 2.0,
    "cisalhamento_seca": 1.5,
}

nome_referencia = "Handroanthus serratifolius = Tabebuia serratifolia"
especie_referencia = base[base["nome_cientifico"] == nome_referencia]

if especie_referencia.empty:
    raise ValueError(f"Espécie não encontrada: {nome_referencia}")

vetor_referencia = dados_normalizados.loc[especie_referencia.index[0]]
diferencas = dados_normalizados - vetor_referencia

# A raiz ponderada evita que variáveis com valores maiores dominem o resultado.
distancia = np.sqrt(
    sum(
        pesos[coluna] * diferencas[coluna] ** 2
        for coluna in colunas
    )
)

resultado = base[
    ["nome_cientifico", "nome_popular_1", "familia", "densidade_basica"]
].copy()
resultado["distancia"] = distancia
resultado = (
    resultado[resultado["nome_cientifico"] != nome_referencia]
    .sort_values("distancia")
)

resultado.head(10)

,nome_cientifico,nome_popular_1,familia,densidade_basica,distancia
2,Bowdichia nitida,SUCUPIRA,Fabaceae,0.77,1.981776
19,Myrocarpus frondosus,CABREÚVA-PARDA,Fabaceae,0.78,2.126197
240,Terminalia amazonia,TANIMBUCA,Combretaceae,0.80,2.610895
250,Maclura tinctoria,AMORA / AMOREIRA,Moraceae,0.73,2.752812
58,Lecythis pisonis subsp. usitata,SAPUCAIA / SAPUCAIA-VERMELHA,Lecythidaceae,0.84,2.802304
146,Buchenavia sp,TANIMBUCA,Combretaceae,0.72,2.827334
265,Pouteria oblanceolata,TUTURUBÁ,Sapotaceae,0.79,2.918872
89,Trichilia lecointei,PRACUÚBA-DA-TERRA-FIRME,Meliaceae,0.90,3.125004
205,Buchenavia grandis = Buchenavia huberi,CUIARANA,Combretaceae,0.79,3.251652
111,Peltogyne paniculata,rosaDINHO,Sapotaceae,0.81,3.275446
